In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

folder = Path("antares_data_clean_from_20260527")

loci_dir = folder / "loci"
alerts_dir = folder / "alerts"


# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

loci = pd.read_parquet(loci_dir)
alerts = pd.read_parquet(alerts_dir)

print(f"Loci rows   : {len(loci):,}")
print(f"Alert rows  : {len(alerts):,}")
print()
print("Locus columns:")
print(loci.columns.tolist())
print()
print("Alert columns:")
print(alerts.columns.tolist())

In [ ]:
# ------------------------------------------------------------
# Parse alert_properties
# ------------------------------------------------------------

def parse_json(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}


props = [parse_json(x) for x in alerts["alert_properties"]]


def get_property(key):
    return np.array([p.get(key, np.nan) for p in props])


alerts["band"] = get_property("lsst_diaSource_band")

alerts["psfFlux"] = pd.to_numeric(
    get_property("lsst_diaSource_psfFlux"),
    errors="coerce",
)

alerts["psfFluxErr"] = pd.to_numeric(
    get_property("lsst_diaSource_psfFluxErr"),
    errors="coerce",
)


# signal-to-noise
alerts["snr"] = alerts["psfFlux"] / alerts["psfFluxErr"]


# AB magnitude
# LSST psfFlux is in nJy:
# m_AB = 31.4 - 2.5 log10(f_nJy)

good_flux = alerts["psfFlux"] > 0

alerts["mag"] = np.nan
alerts.loc[good_flux, "mag"] = (
    31.4
    - 2.5 * np.log10(alerts.loc[good_flux, "psfFlux"])
)


# ------------------------------------------------------------
# Add locus RA / Dec to every alert
# ------------------------------------------------------------

coords = loci[["locus_id", "ra", "dec"]]

alerts = alerts.merge(
    coords,
    on="locus_id",
    how="left",
    validate="many_to_one",
)


# Convert MJD -> date
# MJD 40587 = Unix epoch 1970-01-01

alerts["date"] = pd.to_datetime(
    alerts["mjd"] - 40587,
    unit="D",
    origin="unix",
)

In [ ]:
print("\nBands:")
print(alerts["band"].value_counts(dropna=False))

print("\nMJD range:")
print(alerts["mjd"].min(), alerts["mjd"].max())

print("\nDate range:")
print(alerts["date"].min(), alerts["date"].max())

print("\nMagnitude range:")
print(alerts["mag"].describe())

In [ ]:
# ------------------------------------------------------------
# Helper for sky coordinates
# ------------------------------------------------------------

def sky_xy(ra, dec):
    """
    Convert RA/Dec in degrees to Mollweide coordinates.
    RA increases toward the left, as usual in astronomy.
    """
    ra_wrap = (np.asarray(ra) + 180) % 360 - 180

    x = -np.deg2rad(ra_wrap)
    y = np.deg2rad(dec)

    return x, y


# ------------------------------------------------------------
# Loci + alerts
# ------------------------------------------------------------

fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(111, projection="mollweide")

x_locus, y_locus = sky_xy(loci["ra"], loci["dec"])
x_alert, y_alert = sky_xy(alerts["ra"], alerts["dec"])

ax.scatter(
    x_locus,
    y_locus,
    s=8,
    alpha=0.7,
    label="Loci",
)

ax.scatter(
    x_alert,
    y_alert,
    s=2,
    alpha=0.15,
    label="Alerts",
)

ax.grid(alpha=0.3)
ax.legend()

ax.set_title(
    f"Sky distribution: {len(loci):,} loci, "
    f"{len(alerts):,} alerts"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

def sky_xy(ra, dec):
    ra_wrap = (np.asarray(ra) + 180) % 360 - 180
    x = -np.deg2rad(ra_wrap)
    y = np.deg2rad(dec)
    return x, y

# locus_stats should already contain:
# ['locus_id', 'ra', 'dec', 'n_alerts']

d = locus_stats.copy()
d = d[d["n_alerts"] > 0].copy()

x, y = sky_xy(d["ra"], d["dec"])

# useful percentiles
p90 = np.percentile(d["n_alerts"], 90)
p99 = np.percentile(d["n_alerts"], 99)

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection="mollweide")

# 1) draw everything lightly in gray
#ax.scatter(
#    x, y,
#    s=8,
#    color="lightgray",
#    alpha=0.5,
#    linewidths=0,
#)

# 2) overlay again with log-colored points
sc = ax.scatter(
    x, y,
    c=d["n_alerts"],
    s=10 + 6*np.sqrt(d["n_alerts"]),
    norm=LogNorm(vmin=1, vmax=max(2, p99)),
    alpha=0.85,
    linewidths=0,
)

cb = plt.colorbar(sc, ax=ax, orientation="horizontal", pad=0.08)
cb.set_label("Number of alerts per locus (log color scale)")

ax.grid(alpha=0.3)
ax.set_title("Sky distribution of loci, colored by number of alerts")

plt.tight_layout()
plt.show()

In [ ]:
d = locus_stats.copy()
d = d[d["n_alerts"] > 0].copy()

x, y = sky_xy(d["ra"], d["dec"])

p90 = np.percentile(d["n_alerts"], 90)
p95 = np.percentile(d["n_alerts"], 95)
p99 = np.percentile(d["n_alerts"], 99)

m90 = d["n_alerts"] >= p90
m95 = d["n_alerts"] >= p95
m99 = d["n_alerts"] >= p99

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection="mollweide")

# all loci with alerts
ax.scatter(
    x, y,
    s=8,
    color="lightgray",
    alpha=0.5,
    linewidths=0,
    label="All loci with alerts",
)

# top 10%
ax.scatter(
    x[m90], y[m90],
    s=18,
    alpha=0.7,
    label=f"Top 10% (≥ {p90:.1f} alerts)",
)

# top 5%
ax.scatter(
    x[m95], y[m95],
    s=35,
    alpha=0.8,
    label=f"Top 5% (≥ {p95:.1f} alerts)",
)

# top 1%
ax.scatter(
    x[m99], y[m99],
    s=70,
    alpha=0.95,
    label=f"Top 1% (≥ {p99:.1f} alerts)",
)

ax.grid(alpha=0.3)
ax.legend(loc="lower left")
ax.set_title("Sky distribution with high-alert loci highlighted")

plt.tight_layout()
plt.show()

In [ ]:
alerts_per_locus = alerts.groupby("locus_id").size()

plt.figure(figsize=(4, 3))
plt.hist(alerts_per_locus, bins=30, histtype='step')
plt.yscale("log")
plt.xlabel("Number of alerts per locus")
plt.ylabel("Number of loci")
plt.title("Distribution of alerts per locus")
plt.tight_layout()
plt.show()

In [ ]:
threshold = 5   # try 3, 5, 10

d = locus_stats[locus_stats["n_alerts"] >= threshold].copy()
x, y = sky_xy(d["ra"], d["dec"])

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection="mollweide")

ax.scatter(
    x, y,
    s=20 + 8*np.sqrt(d["n_alerts"]),
    c=d["n_alerts"],
    norm=LogNorm(vmin=threshold, vmax=d["n_alerts"].max()),
    alpha=0.9,
    linewidths=0,
)

ax.grid(alpha=0.3)
ax.set_title(f"Loci with at least {threshold} alerts")

plt.tight_layout()
plt.show()

In [ ]:
bands = ["u", "g", "r", "i", "z", "y"]

available_bands = [
    b for b in bands
    if b in set(alerts["band"].dropna().astype(str))
]

n = len(available_bands)

ncols = 2
nrows = int(np.ceil(n / ncols))

fig = plt.figure(figsize=(8, 4.5 * nrows))

for i, band in enumerate(available_bands):

    ax = fig.add_subplot(
        nrows,
        ncols,
        i + 1,
        projection="mollweide",
    )

    d = alerts[alerts["band"].astype(str) == band]

    x, y = sky_xy(d["ra"], d["dec"])

    ax.scatter(
        x,
        y,
        s=1,
        alpha=0.4,
    )

    ax.grid(alpha=0.3)

    ax.set_title(
        f"{band} band\n{len(d):,} alerts"
    )

plt.subplots_adjust(
    hspace=-0.6,
    wspace=0.1,
)
plt.show()

In [ ]:
band_counts = (
    alerts["band"]
    .value_counts()
    .reindex(bands)
    .dropna()
)

plt.figure(figsize=(7, 4))

plt.bar(
    band_counts.index,
    band_counts.values,
)

plt.xlabel("Band")
plt.ylabel("Number of alerts")
plt.title("Alerts by band")

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(111, projection="mollweide")

x, y = sky_xy(
    alerts["ra"],
    alerts["dec"],
)

date_number = mdates.date2num(alerts["date"])

sc = ax.scatter(
    x,
    y,
    c=date_number,
    s=1,
    alpha=0.2,
)

cb = plt.colorbar(
    sc,
    ax=ax,
    orientation="horizontal",
    pad=0.08,
)

cb.ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y-%m-%d")
)

cb.set_label("Alert observation date")

ax.grid(alpha=0.3)
ax.set_title("Alert observation date across the sky")

plt.tight_layout()
plt.show()

In [ ]:
daily = (
    alerts
    .set_index("date")
    .resample("1D")
    .size()
)

plt.figure(figsize=(12, 4))

plt.plot(
    daily.index,
    daily.values,
)

plt.xlabel("Date")
plt.ylabel("Number of alerts")
plt.title("Number of alerts per day")

plt.tight_layout()
plt.show()

In [ ]:
daily_band = (
    alerts
    .set_index("date")
    .groupby("band")
    .resample("1D")
    .size()
    .rename("count")
    .reset_index()
)

plt.figure(figsize=(12, 5))

for band in bands:

    d = daily_band[
        daily_band["band"].astype(str) == band
    ]

    if len(d):
        plt.plot(
            d["date"],
            d["count"],
            label=band,
        )

plt.xlabel("Date")
plt.ylabel("Number of alerts")
plt.title("Alerts per day by band")
plt.legend(title="Band")

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Overall magnitude distribution
# ------------------------------------------------------------

mag = alerts["mag"].dropna()

plt.figure(figsize=(8, 5))

plt.hist(
    mag,
    bins=60,
    histtype="step",
    linewidth=1.5,
)

plt.xlabel("PSF magnitude [AB]")
plt.ylabel("Number of alerts")

plt.title(
    f"Magnitude distribution "
    f"(N = {len(mag):,})"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))

for band in bands:

    d = alerts[
        alerts["band"].astype(str) == band
    ]

    if len(d) == 0:
        continue

    plt.scatter(
        d["date"],
        d["mag"],
        s=4,
        alpha=0.3,
        label=band,
    )

plt.gca().invert_yaxis()

plt.xlabel("Date")
plt.ylabel("PSF magnitude [AB]")
plt.title("Magnitude vs observation date")

plt.legend(title="Band")

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Basic quality checks
# ------------------------------------------------------------

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(f"Locus rows                  : {len(loci):,}")
print(f"Unique locus IDs            : {loci['locus_id'].nunique():,}")
print()

print(f"Alert rows                  : {len(alerts):,}")
print(f"Unique alert IDs            : {alerts['alert_id'].nunique():,}")
print()

print(
    f"Alerts with valid RA/Dec    : "
    f"{alerts[['ra', 'dec']].notna().all(axis=1).sum():,}"
)

print(
    f"Alerts with valid band      : "
    f"{alerts['band'].notna().sum():,}"
)

print(
    f"Alerts with valid flux      : "
    f"{alerts['psfFlux'].notna().sum():,}"
)

print(
    f"Alerts with positive flux   : "
    f"{(alerts['psfFlux'] > 0).sum():,}"
)

print(
    f"Alerts with valid magnitude : "
    f"{alerts['mag'].notna().sum():,}"
)

print()

print(
    "Date range:",
    alerts["date"].min(),
    "to",
    alerts["date"].max(),
)

print(
    "MJD range:",
    alerts["mjd"].min(),
    "to",
    alerts["mjd"].max(),
)

In [ ]:
print("Duplicate locus IDs:")
print(loci["locus_id"].duplicated().sum())

print("Duplicate alert IDs:")
print(alerts["alert_id"].duplicated().sum())

print("Alerts without corresponding locus:")
print(alerts["ra"].isna().sum())

print("Alerts before MJD 61187:")
print((alerts["mjd"] < 61187).sum())

In [ ]:
alerts_per_locus = alerts.groupby("locus_id").size()

print(alerts_per_locus.describe(
    percentiles=[
        0.1,
        0.25,
        0.5,
        0.75,
        0.9,
        0.95,
        0.99,
    ]
))

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    alerts_per_locus,
    bins=np.arange(
        0.5,
        alerts_per_locus.max() + 1.5,
        1,
    ),
)

plt.xlabel("Number of alerts per locus")
plt.ylabel("Number of loci")
plt.title("Alerts per locus")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    alerts_per_locus,
    bins=40,
)

plt.yscale("log")

plt.xlabel("Number of alerts per locus")
plt.ylabel("Number of loci")
plt.title("Alerts per locus")

plt.tight_layout()
plt.show()

In [ ]:
alerts["night"] = np.floor(alerts["mjd"]).astype(int)

nights_per_locus = (
    alerts.groupby("locus_id")["night"]
    .nunique()
)

plt.figure(figsize=(4, 3))

plt.hist(
    nights_per_locus,
    bins=np.arange(
        0.5,
        nights_per_locus.max() + 1.5,
        1,
    ),
    histtype='step',
)

plt.xlabel("Number of observing nights")
plt.ylabel("Number of loci")
plt.title("Observing nights per locus")

plt.tight_layout()
plt.show()

In [ ]:
baseline = (
    alerts.groupby("locus_id")["mjd"]
    .agg(lambda x: x.max() - x.min())
)

plt.figure(figsize=(4, 3))

plt.hist(
    baseline,
    bins=np.linspace(0,50,51),
    histtype='step',
)

plt.xlabel("Time baseline [days]")
plt.ylabel("Number of loci")
plt.title("Time baseline (locus MJD max-min) per locus")

plt.tight_layout()
plt.show()

In [ ]:
snr = alerts["snr"]

snr = snr[
    np.isfinite(snr)
]

plt.figure(figsize=(4, 3))

plt.hist(
    snr,
    bins=80,
    range=(-10, 100),
    histtype='step',
)

plt.xlabel("PSF flux / PSF flux error")
plt.ylabel("Number of alerts")
plt.title("Alert S/N distribution")

plt.tight_layout()
plt.show()